In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import cv2

In [6]:
import cv2
import os
import pandas as pd

def load_images_to_dataframe(image_dir):

    data = []


    for file_name in os.listdir(image_dir):
        file_path = os.path.join(image_dir, file_name)
        
        # Check if the file is an image (e.g., jpg, png, etc.)
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        
            img = cv2.imread(file_path)
            
            # Convert the image from BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        
            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
            
        
            alpha = 1.2 
            beta = 20   
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

        
            image_id = int(os.path.splitext(file_name)[0])
            
        
            data.append({
                'image_id': image_id,
                'image_data': img
            })


    df = pd.DataFrame(data)
    return df


In [7]:
df_train = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/train/')
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
df_dev = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/dev')

In [8]:
train_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/train/train.csv')
dev_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/dev/dev.csv')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')

train = pd.merge(train_csv, df_train, on='image_id', how='inner')
dev = pd.merge(dev_csv, df_dev, on='image_id', how='inner')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')


In [14]:
from PIL import Image
from torchvision import transforms
import numpy as np
import pandas as pd

augmented_data = []
cnt = 0


img_augmentations = transforms.Compose([
    transforms.ColorJitter(brightness=0.5),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomPosterize(bits=4),
    
])

for idx, row in train.iterrows():
    image_data = row['image_data']
    text = row['transcriptions']
    label = row['labels']
    image_id = row['image_id']

    if label == 1:
        cnt += 1
        # print(cnt)

        
        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)

        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text 
        })

        
        image = Image.fromarray(image_data.astype('uint8'))
        img_aug = img_augmentations(image)
        img_aug = np.array(img_aug)

        augmented_data.append({
            'image_data': img_aug,
            'labels': label,
            'image_id': image_id,
            'transcriptions': text 
        })


augmented_df = pd.DataFrame(augmented_data)


train = pd.concat([train, augmented_df], ignore_index=True)


In [15]:
train.drop(columns='image_id', inplace=True)
dev.drop(columns='image_id', inplace=True)
test.drop(columns='image_id', inplace=True)

In [16]:
train

,labels,transcriptions,image_data
0,0,Sight Adichifying College Staff Expectation Re...,"[[[21, 20, 32], [21, 20, 24], [21, 21, 20], [2..."
1,1,RUKKu ioh IHL~NW irukkuzingafun irukku PHOTOUR...,"[[[165, 189, 190], [159, 183, 184], [154, 177,..."
2,0,Seven Screen Studio @7screenstudio Considering...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
3,0,During Farewell Girls: Boys: Pogumbodhu andha ...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
4,1,போஸ்ட் செய்தவர்: Posted on Qthanga456 Sharecha...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
...,...,...,...
1701,1,Aunty Tamilexpmemez Dei kanna enaku rombha nad...,"[[[32, 32, 32], [32, 32, 32], [32, 32, 32], [3..."
1702,1,சொல்லுங்க விஜய் சார் வைர நெக்லஸ் குடுத்த தருணம...,"[[[80, 80, 96], [80, 80, 96], [80, 80, 96], [8..."
1703,1,சொல்லுங்க விஜய் சார் வைர நெக்லஸ் குடுத்த தருணம...,"[[[84, 85, 92], [84, 85, 92], [84, 85, 92], [8..."
1704,1,shh rendu nimisham sooththula adichikirenu sol...,"[[[16, 16, 16], [16, 16, 16], [16, 16, 16], [1..."


# Resnet

In [23]:
import torch
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import numpy as np

# Custom Dataset class for images
class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data']  # Image array
        label = self.df.loc[idx, 'labels']

        
        image = Image.fromarray(image_data.astype('uint8'))
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)


transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])


train_dataset = ImageDataset(train, transform=transform)
dev_dataset = ImageDataset(dev, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)

# Load ResNet model
resnet_model = models.resnet50(pretrained=True)
num_classes = len(train['labels'].unique())


resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet_model.parameters(), lr=1e-4)

# Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet_model.to(device)

for epoch in range(20):  
    resnet_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
resnet_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in dev_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s] 


Epoch 1, Loss: 0.3717188977888811
Epoch 2, Loss: 0.11838851851915994
Epoch 3, Loss: 0.07204554099850705
Epoch 4, Loss: 0.06331637214268689
Epoch 5, Loss: 0.07653073796558534
Epoch 6, Loss: 0.047586737376872766
Epoch 7, Loss: 0.027851531994975146
Epoch 8, Loss: 0.042935011868156216
Epoch 9, Loss: 0.06626595196446891
Epoch 10, Loss: 0.027045092442090336
Epoch 11, Loss: 0.027785267300629624
Epoch 12, Loss: 0.010851687530167705
Epoch 13, Loss: 0.013859130358434973
Epoch 14, Loss: 0.021419991221481274
Epoch 15, Loss: 0.021174465263739475
Epoch 16, Loss: 0.02657157116353068
Epoch 17, Loss: 0.09932503793345085
Epoch 18, Loss: 0.060838892456295326
Epoch 19, Loss: 0.02920747637122034
Epoch 20, Loss: 0.023860321251270755
Accuracy: 83.45070422535211%


In [30]:
from sklearn.metrics import classification_report, f1_score
import numpy as np
import torch


resnet_model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []

    for batch in dev_loader:
        # Move the batch to the device
        images = batch[0].to(device) 
        labels = batch[1].to(device)  

        # Forward pass
        outputs = resnet_model(images)
        _, predicted = torch.max(outputs, 1)

        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    
    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)

   
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:")
    print(class_report)


Accuracy: 83.45%
F1-score: 0.7668
Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.93      0.89       210
           1       0.74      0.57      0.64        74

    accuracy                           0.83       284
   macro avg       0.80      0.75      0.77       284
weighted avg       0.83      0.83      0.83       284



In [25]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

# test['transcriptions'] = test['transcriptions'].apply(text_preprocessing)
test

,image_id,transcriptions,image_data
0,335,Me causally ignoring when someone complaining ...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
1,1719,பணத்தின் தத்துவம் வாழ்வதற்குச் செலவு மிகக்குறை...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
2,624,Me taking photos of everyone Them Vaa vandhu n...,"[[[22, 28, 36], [22, 28, 31], [22, 28, 31], [2..."
3,317,Smith Annan pitch la paduthuten cup vangitu va...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
4,1576,Mano PhotoGrid IRAL Bsru பாத்துக்கிட்டே ருக்கல...,"[[[115, 80, 63], [115, 80, 63], [115, 80, 63],..."
...,...,...,...
351,434,Muthu Kutti Just now Ivalo pending task vachut...,"[[[52, 52, 52], [52, 52, 52], [52, 52, 52], [5..."
352,1227,காலையில வச்சேன் மாமியார் இன்னைக்கு வெள்ளி கிழம...,"[[[22, 22, 22], [22, 22, 22], [22, 22, 22], [2..."
353,879,~9 Ist in schoollcollege 0 mme newfriend paaka...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
354,602,Muthu Kutti Just now Manager/ TL Raise panna b...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."


In [28]:
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import torch
import numpy as np
from PIL import Image
from torchvision import transforms


class TestImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data'] 
        image = Image.fromarray(image_data.astype('uint8'))

        if self.transform:
            image = self.transform(image)

        return image

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])


test_dataset = TestImageDataset(test, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16)


resnet_model.eval()

test_predictions = []

with torch.no_grad():
    for batch in test_loader:
    
        images = batch.to(device)

    
        outputs = resnet_model(images)
        _, predicted = torch.max(outputs, 1)

    
        test_predictions.extend(predicted.cpu().numpy())

test_predictions = np.array(test_predictions)

test['predictions'] = test_predictions

predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('ResNet_Predictions.csv', index=False)
print("Predictions saved")


Predictions saved


# EfficientNet

In [31]:
import torch
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image


class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data'] 
        label = self.df.loc[idx, 'labels']

       
        image = Image.fromarray(image_data.astype('uint8'))
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)


transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  
])


train_dataset = ImageDataset(train, transform=transform)
dev_dataset = ImageDataset(dev, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)

# Load EfficientNet model
efficientnet_model = models.efficientnet_b0(pretrained=True)
num_classes = len(train['labels'].unique())

# Modify the fully connected layer for our number of classes
efficientnet_model.classifier[1] = nn.Linear(efficientnet_model.classifier[1].in_features, num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(efficientnet_model.parameters(), lr=1e-4)

# Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
efficientnet_model.to(device)

for epoch in range(20):
    efficientnet_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = efficientnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Evaluation
efficientnet_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in dev_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = efficientnet_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total}%")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 127MB/s] 


Epoch 1, Loss: 0.493160386230344
Epoch 2, Loss: 0.2452531859417942
Epoch 3, Loss: 0.1304802872484254
Epoch 4, Loss: 0.07546127236787682
Epoch 5, Loss: 0.059187242147625885
Epoch 6, Loss: 0.04366722491038876
Epoch 7, Loss: 0.051255448273095854
Epoch 8, Loss: 0.04572269191887125
Epoch 9, Loss: 0.030281446853469812
Epoch 10, Loss: 0.01616466780210558
Epoch 11, Loss: 0.022630790263745135
Epoch 12, Loss: 0.01966395487989273
Epoch 13, Loss: 0.024908004378384224
Epoch 14, Loss: 0.017539732288743292
Epoch 15, Loss: 0.027526146628639322
Epoch 16, Loss: 0.025486464711465377
Epoch 17, Loss: 0.018954401202178908
Epoch 18, Loss: 0.0155620772854499
Epoch 19, Loss: 0.018331965427355032
Epoch 20, Loss: 0.022242534065944966
Accuracy: 80.98591549295774%


In [32]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


efficientnet_model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []

    for images, labels in dev_loader:
        # Move data to the device
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = efficientnet_model(images)
        _, predicted = torch.max(outputs, 1)

        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    
    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)


    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:")
    print(class_report)


Accuracy: 80.99%
F1-score: 0.7182
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88       210
           1       0.71      0.46      0.56        74

    accuracy                           0.81       284
   macro avg       0.77      0.70      0.72       284
weighted avg       0.80      0.81      0.80       284



In [34]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')

test

,image_id,transcriptions,image_data
0,335,Me causally ignoring when someone complaining ...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
1,1719,பணத்தின் தத்துவம் வாழ்வதற்குச் செலவு மிகக்குறை...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
2,624,Me taking photos of everyone Them Vaa vandhu n...,"[[[22, 28, 36], [22, 28, 31], [22, 28, 31], [2..."
3,317,Smith Annan pitch la paduthuten cup vangitu va...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
4,1576,Mano PhotoGrid IRAL Bsru பாத்துக்கிட்டே ருக்கல...,"[[[115, 80, 63], [115, 80, 63], [115, 80, 63],..."
...,...,...,...
351,434,Muthu Kutti Just now Ivalo pending task vachut...,"[[[52, 52, 52], [52, 52, 52], [52, 52, 52], [5..."
352,1227,காலையில வச்சேன் மாமியார் இன்னைக்கு வெள்ளி கிழம...,"[[[22, 22, 22], [22, 22, 22], [22, 22, 22], [2..."
353,879,~9 Ist in schoollcollege 0 mme newfriend paaka...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
354,602,Muthu Kutti Just now Manager/ TL Raise panna b...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."


In [35]:
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data']
        image = Image.fromarray(image_data.astype('uint8'))  

        if self.transform:
            image = self.transform(image)

        return image


from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])


test_dataset = TestDataset(test, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16)


efficientnet_model.eval()


test_predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        # Predict
        outputs = efficientnet_model(images)
        _, predicted = torch.max(outputs, 1)

        # Collect predictions
        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions


predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('EfficientNet_Predictions.csv', index=False)
print("Predictions saved")


Predictions saved


# ViT


In [20]:
import torch
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import pandas as pd


class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data']  # Image array
        label = self.df.loc[idx, 'labels']

       
        image = Image.fromarray(image_data.astype('uint8'))
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)


transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  
])


train_dataset = ImageDataset(train, transform=transform)
dev_dataset = ImageDataset(dev, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)


vit_model = models.vit_b_16(pretrained=True) 
num_classes = len(train['labels'].unique())

vit_model.heads = nn.Sequential(
    nn.Linear(vit_model.heads[0].in_features, num_classes)
)





criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vit_model.parameters(), lr=1e-4)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vit_model.to(device)


num_epochs = 20
for epoch in range(num_epochs):
    vit_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = vit_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
vit_model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in dev_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = vit_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Validation Accuracy: {accuracy:.2f}%")


Epoch 1/20, Loss: 0.5206
Epoch 2/20, Loss: 0.3364
Epoch 3/20, Loss: 0.1464
Epoch 4/20, Loss: 0.1225
Epoch 5/20, Loss: 0.0931
Epoch 6/20, Loss: 0.0678
Epoch 7/20, Loss: 0.0724
Epoch 8/20, Loss: 0.0584
Epoch 9/20, Loss: 0.0198
Epoch 10/20, Loss: 0.0198
Epoch 11/20, Loss: 0.0057
Epoch 12/20, Loss: 0.0030
Epoch 13/20, Loss: 0.0022
Epoch 14/20, Loss: 0.0017
Epoch 15/20, Loss: 0.0018
Epoch 16/20, Loss: 0.0015
Epoch 17/20, Loss: 0.0017
Epoch 18/20, Loss: 0.0039
Epoch 19/20, Loss: 0.0849
Epoch 20/20, Loss: 0.1251
Validation Accuracy: 75.00%


In [21]:
from sklearn.metrics import classification_report, f1_score
import numpy as np


vit_model.eval()
with torch.no_grad():
    y_true = []
    y_pred = []

    for images, labels in dev_loader:
        # Move data to the device
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = vit_model(images) 
        _, predicted = torch.max(outputs, 1) 

        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

    
    accuracy = 100 * sum(np.array(y_true) == np.array(y_pred)) / len(y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    class_report = classification_report(y_true, y_pred)

    
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:")
    print(class_report)


Accuracy: 75.00%
F1-score: 0.5912
Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.93      0.85       210
           1       0.55      0.24      0.34        74

    accuracy                           0.75       284
   macro avg       0.66      0.59      0.59       284
weighted avg       0.72      0.75      0.71       284



In [22]:
df_test = load_images_to_dataframe('/kaggle/input/tamillang/Misogyny Meme Detection/test')
test_csv = pd.read_csv('/kaggle/input/tamillang/Misogyny Meme Detection/test/test.csv')
test = pd.merge(test_csv, df_test, on='image_id', how='inner')


test

,image_id,transcriptions,image_data
0,335,Me causally ignoring when someone complaining ...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
1,1719,பணத்தின் தத்துவம் வாழ்வதற்குச் செலவு மிகக்குறை...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."
2,624,Me taking photos of everyone Them Vaa vandhu n...,"[[[22, 28, 36], [22, 28, 31], [22, 28, 31], [2..."
3,317,Smith Annan pitch la paduthuten cup vangitu va...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
4,1576,Mano PhotoGrid IRAL Bsru பாத்துக்கிட்டே ருக்கல...,"[[[115, 80, 63], [115, 80, 63], [115, 80, 63],..."
...,...,...,...
351,434,Muthu Kutti Just now Ivalo pending task vachut...,"[[[52, 52, 52], [52, 52, 52], [52, 52, 52], [5..."
352,1227,காலையில வச்சேன் மாமியார் இன்னைக்கு வெள்ளி கிழம...,"[[[22, 22, 22], [22, 22, 22], [22, 22, 22], [2..."
353,879,~9 Ist in schoollcollege 0 mme newfriend paaka...,"[[[20, 20, 20], [20, 20, 20], [20, 20, 20], [2..."
354,602,Muthu Kutti Just now Manager/ TL Raise panna b...,"[[[255, 255, 255], [255, 255, 255], [255, 255,..."


In [23]:
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import pandas as pd
import torch
import numpy as np


class TestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_data = self.df.loc[idx, 'image_data']  # Image array
        image = Image.fromarray(image_data.astype('uint8'))  # Convert to PIL Image

        if self.transform:
            image = self.transform(image)

        return image


from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) 
])


test_dataset = TestDataset(test, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16)


vit_model.eval()

# Store predictions
test_predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        
        outputs = vit_model(images)
        _, predicted = torch.max(outputs, 1)

        
        test_predictions.extend(predicted.cpu().numpy())


test_predictions = np.array(test_predictions)


test['predictions'] = test_predictions


predictions_df = pd.DataFrame({
    'image_id': test['image_id'],
    'predictions': test_predictions
})

predictions_df.to_csv('ViT_Predictions.csv', index=False)
print("Predictions saved")


Predictions saved
